## import lib

In [14]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json
import gradio as gr
from IPython.display import Markdown,display
import sqlite3
import requests
import json

## Create Ollama client

In [2]:
requests.get("http://localhost:11434").content

b'Ollama is running'

In [15]:
ollama_url="http://localhost:11434/v1"
ollama=OpenAI(base_url=ollama_url,api_key="ollama")

# Create SQLite Database

In [16]:
DB="exam.db"
with sqlite3.connect(DB) as conn:
    cursor=conn.cursor()
    cursor.execute("drop table if exists QA")
    cursor.execute("create table if not exists QA (id integer primary key autoincrement, ques text ,ans text)")
    conn.commit()

#  Insert Questions and Answers


In [17]:
def insert_QA(ques_ans):
    with sqlite3.connect(DB) as conn:
        cursor=conn.cursor()
        cursor.executemany("insert into QA (ques,ans) values (?,?)",(ques_ans))
        conn.commit()

In [18]:
Ques_Ans = [

    ("Why is Deep Learning powerful for image recognition?",
     "Because deep neural networks can automatically extract important features such as edges, shapes, and objects from images."),

    ("How does Machine Learning differ from traditional programming?",
     "In traditional programming rules are explicitly written, while in Machine Learning the system learns rules from data."),

    ("Why do we split data into training and testing sets?",
     "To evaluate how well the model generalizes to unseen data and avoid overfitting."),

    ("What happens if the learning rate is too high?",
     "The model may fail to converge and can overshoot the optimal solution during training."),

    ("What happens if the learning rate is too low?",
     "Training becomes very slow and the model may get stuck before reaching the optimal solution."),

    ("Why are activation functions important in neural networks?",
     "They introduce non-linearity which allows neural networks to learn complex relationships in data."),

    ("How does CNN reduce the number of parameters compared to fully connected networks?",
     "CNN uses shared filters and local connections which greatly reduce the number of trainable parameters."),

    ("Why are LSTMs better than standard RNNs for long sequences?",
     "Because LSTMs can remember important information for long periods and reduce the vanishing gradient problem."),

    ("What is the purpose of dropout in deep learning?",
     "Dropout reduces overfitting by randomly disabling some neurons during training."),

    ("Why is normalization important in Machine Learning?",
     "Normalization scales features to similar ranges which improves training stability and convergence speed."),

    ("How does backpropagation improve a neural network?",
     "It calculates errors and updates weights to minimize the loss function."),

    ("What is the role of an optimizer in deep learning?",
     "The optimizer updates model parameters to reduce prediction error during training."),

    ("Why is GPU commonly used in Deep Learning?",
     "Because GPUs can process many calculations in parallel which speeds up neural network training."),

    ("How does transfer learning save training time?",
     "It uses knowledge from a pre-trained model so the network does not need to learn from scratch."),

    ("What is the difference between classification and regression?",
     "Classification predicts categories while regression predicts continuous numerical values."),

    ("Why can Deep Learning require large datasets?",
     "Because deep networks have many parameters and need large amounts of data to learn effectively."),

    ("How does data augmentation help in training?",
     "It increases dataset diversity by modifying existing data which improves model generalization."),

    ("What is the vanishing gradient problem?",
     "It occurs when gradients become extremely small during backpropagation making learning difficult in deep networks."),

    ("Why is cross-validation useful in Machine Learning?",
     "It provides a more reliable evaluation of model performance using different subsets of data."),

    ("How does attention mechanism improve deep learning models?",
     "Attention helps the model focus on the most important parts of the input data during prediction.")
]

In [19]:
insert_QA(Ques_Ans)

# Fetch Random Questions

In [20]:
def get_ques_ans():
    with sqlite3.connect(DB) as conn:
        cursor=conn.cursor()
        cursor.execute("select ques,ans from QA order by random() limit 3")
        row=cursor.fetchall()
        return row

In [21]:
get_ques_ans()


[('Why are LSTMs better than standard RNNs for long sequences?',
  'Because LSTMs can remember important information for long periods and reduce the vanishing gradient problem.'),
 ('What is the difference between classification and regression?',
  'Classification predicts categories while regression predicts continuous numerical values.'),
 ('How does Machine Learning differ from traditional programming?',
  'In traditional programming rules are explicitly written, while in Machine Learning the system learns rules from data.')]

# Initialize exam variables

In [22]:
#number of question for each student
num_quest=3
#question and correct answer
ques_crans=get_ques_ans()   #quetsion and correct answe
ques_id=0
#student answer
student_answer=[]

# Chat function

In [23]:
def chat(message,history):

    global ques_id
    global student_answer

    questions_data = []
    print(message)

    # save the student answer
    if ques_id > 0:
        student_answer.append(message)

    # ask the next question
    if ques_id < len(ques_crans):

        question = ques_crans[ques_id][0]

        ques_id += 1
        history+=[{"role":"assistant","content":question}]

        return history

    # get the score
    elif ques_id == len(ques_crans):

        for i, element in enumerate(ques_crans):

            questions_data.append({
                "question": element[0],
                "correct_answer": element[1],
                "student_answer": student_answer[i]
            })

        print(questions_data)

        prompt = f"""
            Evaluate the following student answers.

                Data:
                   {json.dumps(questions_data, indent=2)}

                   Return ONLY valid JSON in this format:

                     {{
                        "results": [
                             {{
                                "question": "...",
                                "score": 0-10,
                                "is_correct": true,
                                "feedback": "short feedback"
                               }}
                          ],

                        "final_score": 0-10
                     }}
        """


        response = ollama.chat.completions.create(
            model="gemma4",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        ques_id += 1

         
        reply=response.choices[0].message.content
        history+=[{"role":"assistant","content":reply}]
        return history

    else:
        history+=[{"role":"assistant","content":"End of exam"}]
        return history

# Chatbot UI

In [ ]:
def get_chatbot_msg(msg,history):
   
    history = history or []

    history=history+[{"role": "user", "content": msg}]

    return "",msg,history 

with gr.Blocks(title="AI Exam Chatbot") as demo:

    gr.Markdown("# 🧠 AI Exam Chatbot")

    chatbot = gr.Chatbot(type="messages")

    msg = gr.Textbox(label="chat with your AI examiner")
    message = gr.State()
    msg.submit(get_chatbot_msg,inputs=[msg,chatbot],outputs=[msg,message,chatbot]).then(chat,inputs=[message,chatbot],outputs=chatbot)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


START
CNN uses shared filters and local connections which greatly reduce the number of trainable parameters
machine learning need dataset to learn while tradional programming is a set of steps only, with no learning.
i dont know
[{'question': 'How does CNN reduce the number of parameters compared to fully connected networks?', 'correct_answer': 'CNN uses shared filters and local connections which greatly reduce the number of trainable parameters.', 'student_answer': 'CNN uses shared filters and local connections which greatly reduce the number of trainable parameters'}, {'question': 'How does Machine Learning differ from traditional programming?', 'correct_answer': 'In traditional programming rules are explicitly written, while in Machine Learning the system learns rules from data.', 'student_answer': 'machine learning need dataset to learn while tradional programming is a set of steps only, with no learning.'}, {'question': 'Why is cross-validation useful in Machine Learning?', 'corre